# Elastic vibration: image tracking

Companion code for *Mechanical Characterization of 3D-Printed Materials Using Printer-Native Motion*.
Run from top to bottom after installing `../requirements.txt`. Read [README.md](README.md) first.
The implementation lives in `tracking.py`; this notebook contains the experiment configuration.
Use physical camera acquisition FPS (1000 fps in the manuscript), not slow-motion playback FPS.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt

HERE = Path.cwd() if (Path.cwd() / "tracking.py").is_file() else Path.cwd() / "elastic"
if not (HERE / "tracking.py").is_file():
    raise RuntimeError("Start Jupyter in the repository root or the elastic folder")
sys.path.insert(0, str(HERE.resolve()))
from tracking import TrackingConfig, track_video

## 1. Configure one experiment

The legacy profile preserves the last active detector in the supplied notebook.
The component profile retains its alternative morphology and median-tip detector.
Inspect overlays before selecting a free-vibration interval. For a different beam color,
set HSV bounds in OpenCV units: hue 0–179, saturation/value 0–255.

In [ ]:
VIDEO = HERE / "data" / "vibration.mp4"
OUTPUT = HERE / "results" / "tip_tracking.csv"
ACQUISITION_FPS = 1000.0  # Replace with the actual physical acquisition rate.
CONFIG = TrackingConfig(
    profile="legacy",
    hsv_lower=None,
    hsv_upper=None,
    roi=None,             # Optional (left, top, width, height).
    max_jump_px=None,     # Optional rejection threshold; NaN marks suspect frames.
)
PREVIEW_EVERY = 50

## 2. Track all frames

No intermediate JPEGs are needed. Every decoded frame keeps its original index and
timestamp. Failed detections remain in the table. Use a new output name for a rerun.

In [ ]:
tracking = track_video(VIDEO, OUTPUT, acquisition_fps=ACQUISITION_FPS,
                       config=CONFIG, preview_every=PREVIEW_EVERY)
print(tracking["tracking_status"].value_counts())
tracking.head()

## 3. Inspect the response and fit in MATLAB

Inspect saved previews together with this trace. Select a clean free-vibration window
beginning at a displacement extremum, determine its equilibrium offset, and use
`response_fit.m` as documented in the README. The source files supplied for this release
do not include the manuscript's FFT peak-refinement implementation.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(tracking["time_s"], tracking["raw_tip_displacement_px"], alpha=0.5, label="Raw detection")
ax.plot(tracking["time_s"], tracking["tip_displacement_px"], label="Accepted displacement")
ax.set(xlabel="Time (s)", ylabel="Tip displacement (pixel)")
ax.legend()
plt.show()